# Activity: Metabolic Networks and the Stoichiometric Matrix

In this activity, we will explore genome-scale metabolic networks using the [BiGG Models database and API](http://bigg.ucsd.edu/). We will download a model from the BiGG Models database, parse the model information, and construct the stoichiometric matrix for the model.

> __Learning Objectives:__
> 
> By the end of this activity, you will be able to:
> * **Access biological databases via APIs**: Retrieve genome-scale metabolic models from the BiGG Models database using RESTful API calls and parse the structured model data.
> * **Construct and interpret stoichiometric matrices**: Build the stoichiometric matrix from metabolic model data and understand how coefficients represent the participation of metabolites in biochemical reactions.
> * **Analyze metabolic network connectivity**: Compute connectivity matrices from stoichiometric matrices to identify highly connected metabolites and reactions in metabolic networks.

Let's get started!

---

## Background: Metabolic networks and the stoichiometric matrix
A metabolic network encompasses all the chemical reactions associated with metabolism, i.e., the breakdown of raw materials such as sugars ([catabolism](https://en.wikipedia.org/wiki/Catabolism)) and the production of macromolecules, e.g., DNA, RNA, proteins, lipids, etc. ([anabolism](https://en.wikipedia.org/wiki/Anabolism)). These networks are curated for thousands of organisms and are available in various online databases. 

Let's check out a few of these online metabolic databases:
* [Minoru Kanehisa, Miho Furumichi, Yoko Sato, Yuriko Matsuura, Mari Ishiguro-Watanabe, KEGG: biological systems database as a model of the real world, Nucleic Acids Research, Volume 53, Issue D1, 6 January 2025, Pages D672–D677, https://doi.org/10.1093/nar/gkae909](https://academic.oup.com/nar/article/53/D1/D672/7824602)
* [Karp PD, Billington R, Caspi R, Fulcher CA, Latendresse M, Kothari A, Keseler IM, Krummenacker M, Midford PE, Ong Q, Ong WK, Paley SM, Subhraveti P. The BioCyc collection of microbial genomes and metabolic pathways. Brief Bioinform. 2019 Jul 19;20(4):1085-1093. doi: 10.1093/bib/bbx085. PMID: 29447345; PMCID: PMC6781571.](https://pubmed.ncbi.nlm.nih.gov/29447345/)
* [Charles J Norsigian, Neha Pusarla, John Luke McConn, James T Yurkovich, Andreas Dräger, Bernhard O Palsson, Zachary King, BiGG Models 2020: multi-strain genome-scale models and expansion across the phylogenetic tree, Nucleic Acids Research, Volume 48, Issue D1, 08 January 2020, Pages D402–D406, https://doi.org/10.1093/nar/gkz1054](https://academic.oup.com/nar/article/48/D1/D402/5614178)

There are many other databases with information about enzymes and other biological numbers that we may be interested in:
* [Antje Chang, Lisa Jeske, Sandra Ulbrich, Julia Hofmann, Julia Koblitz, Ida Schomburg, Meina Neumann-Schaal, Dieter Jahn, Dietmar Schomburg, BRENDA, the ELIXIR core data resource in 2021: new developments and updates, Nucleic Acids Research, Volume 49, Issue D1, 8 January 2021, Pages D498–D508, https://doi.org/10.1093/nar/gkaa1025](https://academic.oup.com/nar/article/49/D1/D498/5992283)
* [Ron Milo, Paul Jorgensen, Uri Moran, Griffin Weber, Michael Springer, BioNumbers—the database of key numbers in molecular and cell biology, Nucleic Acids Research, Volume 38, Issue suppl_1, 1 January 2010, Pages D750–D753, https://doi.org/10.1093/nar/gkp889](https://academic.oup.com/nar/article/38/suppl_1/D750/3112244)
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

___

## Task 1: Download a genome-scale metabolic model from BiGG Models
In this task, we will download a genome-scale metabolic model from [the BiGG Models database](http://bigg.ucsd.edu/) using [the BiGG Models API](http://bigg.ucsd.edu/data_access).

We developed a simple software development kit (SDK) for [the BiGG Models application programming interface at the University of California, San Diego](http://bigg.ucsd.edu/). The [BiGG Models database](http://bigg.ucsd.edu/) integrates published genome-scale metabolic networks into a single database with standardized nomenclature and structure. 

> __What is it?__ [The BiGG models API](http://bigg.ucsd.edu/data_access) allows users to programmatically access genome-scale stoichiometric model reconstructions using a simple web API. There are `108` models of intracellular biochemistry occurring in various organisms (including humans) in the database (so far); [see here for a list of models](http://bigg.ucsd.edu/models).

We call the model download endpoint of [the BiGG models API](http://bigg.ucsd.edu/data_access) and then save the model file to disk (so we don't hit the API unless we have to). This call returns model information organized as [a Julia dictionary](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) in the `model::Dict{String, Any}` variable. If a model file is saved, we use the cached file instead of making an API call.

In [2]:
model = let

    # build download endpoint -
    baseurl = "http://bigg.ucsd.edu"; # base url to download model
    modelid = "iND750"; # model id to download
    path_to_saved_model_file = joinpath(_PATH_TO_DATA, "saved-model-$(modelid).jld2");

    # check: do we have a model file saved?
    model = nothing;
    if (isfile(path_to_saved_model_file) == false)
        
        endpoint = MyBiggModelsDownloadModelEndpointModel();
        endpoint.bigg_id = modelid;
        url = build(baseurl, endpoint)
        model = MyBiggModelsDownloadModelEndpointModel(url);

        # Before we move on, save this model for later (so we don't keep hitting the API)
        save(path_to_saved_model_file, Dict("model" => model));
    else
        model = load(path_to_saved_model_file)["model"];
    end
    model; # return the model (either saved, or downloaded)
end

JSON.Object{String, Any} with 6 entries:
  "metabolites"  => Any[Object{String, Any}("id"=>"ala__L_e", "name"=>"L-Alanin…
  "reactions"    => Any[Object{String, Any}("id"=>"EX_met__L_e", "name"=>"L-Met…
  "genes"        => Any[Object{String, Any}("id"=>"YER061C", "name"=>"CEM1", "n…
  "id"           => "iND750"
  "compartments" => Object{String, Any}("c"=>"cytosol", "e"=>"extracellular spa…
  "version"      => "1"

__Genes records__: Each model has a set of genes associated with the model's chemical reactions. The `genes` has several subfields that give information about the gene. For example, the `refseq_name` field [is the name in the NCBI reference sequence database](https://www.ncbi.nlm.nih.gov/refseq/).

In [3]:
model["genes"][1]["annotation"]

JSON.Object{String, Any} with 5 entries:
  "ncbigene"         => Any["856790"]
  "refseq_locus_tag" => Any["YER061C"]
  "refseq_name"      => Any["CEM1"]
  "sbo"              => "SBO:0000243"
  "sgd"              => Any["S000000863"]

__Metabolite records__: Each metabolite (chemical compound) in the network has an associated metabolite record with several fields. Let's take a look at the metabolite at index `1`. The `id` field holds an abbreviation or symbol associated with this metabolite.

In [4]:
model["metabolites"][1] # example metabolite record

JSON.Object{String, Any} with 6 entries:
  "id"          => "ala__L_e"
  "name"        => "L-Alanine"
  "compartment" => "e"
  "formula"     => "C3H7NO2"
  "notes"       => Object{String, Any}("original_bigg_ids"=>Any["ala_DASH_L_e"])
  "annotation"  => Object{String, Any}("bigg.metabolite"=>Any["ala__L"], "biocy…

__Reaction records__: Similarly, each reaction in the network has a reaction record with several fields. Let's look at the reaction record at index `25`. The species involved in a reaction are contained in the `metabolites` field, which lists the stoichiometric coefficients associated with this particular reaction.

In [5]:
model["reactions"][156] # example reaction record

JSON.Object{String, Any} with 9 entries:
  "id"                 => "ASP1DC"
  "name"               => "Aspartate 1-decarboxylase"
  "metabolites"        => Object{String, Any}("ala_B_c"=>1.0, "asp__L_c"=>-1.0,…
  "lower_bound"        => 0.0
  "upper_bound"        => 999999.0
  "gene_reaction_rule" => ""
  "subsystem"          => "Pantothenate and CoA Biosynthesis"
  "notes"              => Object{String, Any}("original_bigg_ids"=>Any["ASP1DC"…
  "annotation"         => Object{String, Any}("bigg.reaction"=>Any["ASP1DC"], "…

___

## Task 2: Build a stoichiometric matrix
In this task, we'll build a stoichiometric matrix $\mathbf{S}$ using the metabolite and reaction records that we just downloaded. 

The stoichiometric matrix $\mathbf{S}$ is the digital representation of the biochemistry occurring inside some volume, i.e., inside the cell, in a test tube in the case of cell-free systems, or some abstract volume such as a compartment or pseudo compartment of interest.

Suppose we have a set of biochemical reactions $\mathcal{R}$ involving chemical species (metabolite) set $\mathcal{M}$. Then, the stoichiometric matrix is a $\mathbf{S}\in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}$ matrix, where $|\mathcal{M}|$ denotes the number of chemical species and $|\mathcal{R}|$ denotes the number of reactions. The elements of the stoichiometric matrix $\sigma_{ij}\in\mathbf{S}$ are stoichiometric coefficients  such that:
* $\sigma_{ij}>0$: Chemical species (metabolite) $i$ is _produced_ by reaction $j$. Species $i$ is a product of reaction $j$.
* $\sigma_{ij} = 0$: Chemical species (metabolite) $i$ is not connected with reaction $j$
* $\sigma_{ij}<0$: Chemical species (metabolite) $i$ is _consumed_ by reaction $j$. Species $i$ is a reactant of reaction $j$.


We'll build $\mathbf{S}$ using nested [`for` loops](https://docs.julialang.org/en/v1/base/base/#for):

> __How it works:__
> 
> * _In the outer loop_: we'll iterate over the system's `metabolites` (chemical species) and select the `id` field from the `metabolites` record.
> * _In the inner loop_: we iterate over each reaction. For each reaction record, we ask if this reaction has an entry for the current metabolite `id` value; if it does, we grab the stoichiometric coefficient $\sigma_{ij}\in\mathbf{S}$ corresponding to this metabolite and reaction.

We'll save the stoichiometric matrix in the `S::Array{Float64, 2}` variable. After building the stoichiometric matrix, we can check its size using the [`size(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.size).

In [6]:
S = let

    # get some data from the model -
    m = model["metabolites"]; # get list of metabolites
    r = model["reactions"]; # get list of reactions
    number_of_rows = length(m); # how many metabolites do we have? (rows)
    number_of_cols = length(r); # how many reactions do we have? (cols)
    S = zeros(number_of_rows,number_of_cols); # initialize an empty stoichiometric matrix

    # let's build a stm -
    for i ∈ eachindex(m)
        metabolite = m[i]["id"]; # we are checking if this metabolite is in the reaction record
        for j ∈ eachindex(r)
            reaction = r[j];
            if (haskey(reaction["metabolites"], metabolite) == true)
                S[i,j] = reaction["metabolites"][metabolite];
            end
        end
    end
    S; 
end;

How many metabolites and reactions are in this model? 

In [7]:
size(S)

(1059, 1266)

Finally, in the analysis below, we'll use the __binary version__ of the stoichiometric matrix, where all non-zero entries are set to `1`. That is, if a metabolite is connected to a reaction (either as a reactant or product), the corresponding entry in the binary stoichiometric matrix is `1`; otherwise, it is `0`.

Let's compute the binary stoichiometric matrix $\bar{\mathbf{S}}$ by [calling the `binary(...)` method](src/Compute.jl). 

In [8]:
S̄ = binary(S) # convert all non-zero entries to 1

1059×1266 Matrix{Int64}:
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 ⋮              ⋮              ⋮        ⋱     ⋮              ⋮              ⋮
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0 

___

## Task 3: Analyze the connectivity of a metabolic network
In this task, we will analyze the connectivity of the metabolic network using the stoichiometric matrix $\mathbf{S}$ that we built in Task 2.

The connectivity of metabolites in a metabolic network provides insight into the importance of a metabolite or reaction. Altering the enzyme level of a highly connected reaction or conditions affecting a highly connected metabolite may yield a greater response than changing an unconnected one. 

> __What is it?__ We can explore the connectivity of the metabolism of an organism by constructing connectivity matrices from the binary stoichiometric matrices. In particular, we compute the metabolite connectivity array $\mathbf{C}_{m}$ or the reaction connectivity matrix $\mathbf{C}_{r}$ and look at some of its properties. 

Let's start with the metabolite connectivity array $\mathbf{C}_{m}$.

### Metabolite connectivity
The metabolite connectivity matrix, defined as $\mathbf{C}_{m} \equiv \bar{\mathbf{S}}\bar{\mathbf{S}}^{\top}$, is an $|\mathcal{M}|\times|\mathcal{M}|$ symmetric array with the following features:
* __Diagonal elements__: The elements along the central diagonal $c_{ii}\in\mathbf{C}_{m}$ are the total number of reactions a particular metabolite participates in. However, because we removed the directionality when we computed the binary stoichiometric matrix, we have no information about whether the participation is a reactant or product.
* __Off diagonal elements__: The off-diagonal elements $c_{ij}\in\mathbf{C}_{m}$ where $i\neq{j}$ describe how many reactions metabolite $i$ has in common with metabolite $j$, i.e., the number of joint reactions for the pair.

Let's store the metabolite connectivity matrix in the `Cₘ::Array{Int64, 2}` variable:

In [9]:
Cₘ = S̄*transpose(S̄) # metabolite connectivity matrix M x M matrix

1059×1059 Matrix{Int64}:
 2  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  2  0  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  2  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  2  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  2  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  2  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  2  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  3  1  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  1  3  0  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  2  0  0  0     0  0  0  0  0  0  0  0  0  0  0  0
 ⋮              ⋮              ⋮        ⋱           ⋮              ⋮        
 0  0  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  2  1  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  

In [10]:
argmax(diag(Cₘ)) |> i-> model["metabolites"][i] # maximum connectivity metabolite

JSON.Object{String, Any} with 6 entries:
  "id"          => "h_c"
  "name"        => "H+"
  "compartment" => "c"
  "formula"     => "H"
  "notes"       => Object{String, Any}("original_bigg_ids"=>Any["h_c"])
  "annotation"  => Object{String, Any}("bigg.metabolite"=>Any["h"], "biocyc"=>A…

Let's sort the diagonal elements $\text{diag}(\mathbf{C}_{m})$ from largest to smallest and then build [a table using the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl). Unhide the code block below to see how we constructed the metabolite connectivity table.

> __Summary__: Several of the highest connected metabolites are associated with energy metabolism. Thus, if we manipulate [the energy metabolic pathways](https://www.genome.jp/kegg/pathway.html#energy) or otherwise perturb the energetics of the cell, we should expect a significant (for better or worse) response from the system. Interesting! But what about the reaction connectivity?

Which chemical species (metabolites) are the most connected in this metabolic network?

In [11]:
let
    df = DataFrame();
    d = diag(Cₘ);
    î = sortperm(d, rev=true);
    number_of_rows_in_table = 20;
    for i ∈ î[1:number_of_rows_in_table]
        m = model["metabolites"][i]
        row_df = (
            index = i,
            compartment = m["compartment"],
            name = m["name"],
            id = m["id"],
            connections = d[i]
        );
        push!(df, row_df) # capture the row
    end

    # make a table -
    pretty_table(
        df;
        fit_table_in_display_vertically = false,
        fit_table_in_display_horizontally = false,
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

 ------- ------------- ------------------------------------------------------- ---------- -------------
  index   compartment                                                    name         id   connections 
  Int64        String                                                  String     String         Int64 
 ------- ------------- ------------------------------------------------------- ---------- -------------
    240             c                                                      H+        h_c           435
    233             c                                                 H2O H2O      h2o_c           222
    278             c                                       ATP C10H12N5O13P3      atp_c           143
    153             m                                                      H+        h_m           106
    101             c                                       ADP C10H12N5O10P2      adp_c           102
    164             c                                               P

### Reaction connectivity
The reaction connectivity matrix, defined as $\mathbf{C}_{r} \equiv \bar{\mathbf{S}}^{\top}\bar{\mathbf{S}}$, is an $|\mathcal{R}|\times|\mathcal{R}|$ symmetric array with the following features:
* __Diagonal elements__: The elements along the central diagonal $c_{ii}\in\mathbf{C}_{r}$ are the total number of reactants and products of a particular reaction. However, because we removed the directionality when we computed the binary stoichiometric matrix, we have no information about the number of reactants or products, just the total participation number for a reaction.
* __Off diagonal elements__: The off-diagonal elements of the reaction matrix $c_{ij}\in\mathbf{C}_{r}$ where $i\neq{j}$ describe how many metabolites are shared between reaction $i$ and $j$, i.e., the number of joint metabolites for the pair.

We store the reaction connectivity matrix in the `Cᵣ::Array{Int64, 2}` variable:

In [12]:
Cᵣ = transpose(S̄)*S̄ # metabolite connectivity matrix R x R matrix

1266×1266 Matrix{Int64}:
 1  0  0  0  0  0  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0   0  0
 0  1  0  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  1  0  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  1  0  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  1  0  0  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  0  1  0  0  0  0  0  0  …  0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  0  0  1  0  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  0  0  0  1  0  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  0  0  0  0  1  0  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 0  0  0  0  0  0  0  0  0  1  0  0     0  0  0  0  0  0  0  0  0  0   0  0
 ⋮              ⋮              ⋮     ⋱     ⋮              ⋮               ⋮
 0  0  0  0  0  0  0  0  0  0  0  0     0  0  1  5  1  2  0  0  0  0   3  0
 0  0  0  0  0  0  0  0  0  0  0  0     0  0  1  1  5  2  1  1 

What is the most connected reaction? Find the index of the maximum diagonal element [using the `argmax(...)` method](https://docs.julialang.org/en/v1/base/collections/#Base.argmax), and then [pipe that index using the `|>` operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) to the reaction array. Save the highest connected reaction in the `most_connected_reaction::Dict{String,Any}` variable:

In [13]:
most_connected_reaction = argmax(diag(Cᵣ)) |> i-> model["reactions"][i] # maximum connectivity reaction

JSON.Object{String, Any} with 10 entries:
  "id"                    => "BIOMASS_SC4_bal"
  "name"                  => "Biomass SC4 bal"
  "metabolites"           => Object{String, Any}("13BDglcn_c"=>-1.1348, "adp_c"…
  "lower_bound"           => 0.0
  "upper_bound"           => 999999.0
  "gene_reaction_rule"    => ""
  "objective_coefficient" => 1.0
  "subsystem"             => "Biomass and maintenance functions"
  "notes"                 => Object{String, Any}("original_bigg_ids"=>Any["biom…
  "annotation"            => Object{String, Any}("bigg.reaction"=>Any["BIOMASS_…

We can [call the `reactionstring(...)` method](src/Compute.jl) with the `metabolites` dictionary from a `reaction` dictionary to see the reaction string: 

In [14]:
test = reactionstring(most_connected_reaction["metabolites"])

"4.5e-5 pe_SC_c + 0.0599 ump_c + 0.0024 dcmp_c + 0.2862 lys__L_c + 0.0663 his__L_c + 5.3e-5 ptd1ino_SC_c + 0.046 gmp_c + 0.046 amp_c + 0.0234 tre_c + 6.6e-5 triglyc_SC_c + 0.1927 ile__L_c + 0.102 tyr__L_c + 0.2904 gly_c + 0.1914 thr__L_c + 59.276 atp_c + 0.1647 pro__L_c +" ⋯ 239 bytes ⋯ "+ 0.1607 arg__L_c + 0.5185 glycogen_c + 1.7e-5 ps_SC_c + 0.1054 gln__L_c + 0.0284 trp__L_c + 0.4588 ala__L_c + 0.0066 cys__L_c + 0.0024 dgmp_c + 0.0036 dtmp_c + 0.1017 asn__L_c + 59.276 h2o_c + 0.2975 asp__L_c + 0.0507 met__L_c = 59.276 adp_c + 59.305 pi_c + 58.7162 h_c"

Let's sort the diagonal elements $\text{diag}(\mathbf{C}_{r})$ from largest to smallest and then build [a table using the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl). Unhide the code block below to see how we constructed the reaction connectivity table.

> __Summary__: The most connected reaction list makes intuitive sense (or maybe not) depending on the network we are looking at. For example, for networks with a `biomass` reaction (which describes the requirements to make more cells), we would intuitively expect those reactions to be highly ranked. However, for other networks that describe non-replicating systems, the importance of the reaction may be specific to the function of the cell.

Which reactions are the most connected in this metabolic network?

In [16]:
let
    df = DataFrame();
    d = diag(Cᵣ);
    î = sortperm(d, rev=true);
    number_of_rows_in_table = 20;
    for i ∈ î[1:number_of_rows_in_table]
        m = model["reactions"][i]
        row_df = (
            index = i,
            id = m["id"],
            connections = d[i],
            reaction = reactionstring(m["metabolites"]),
        );
        push!(df, row_df) # capture the row
    end
  
    df; # show dataframe
end

Row,index,id,connections,reaction
,Int64,String,Int64,String
1,1265,BIOMASS_SC4_bal,46,4.5e-5 pe_SC_c + 0.0599 ump_c + 0.0024 dcmp_c + 0.2862 lys__L_c + 0.0663 his__L_c + 5.3e-5 ptd1ino_SC_c + 0.046 gmp_c + 0.046 amp_c + 0.0234 tre_c + 6.6e-5 triglyc_SC_c + 0.1927 ile__L_c + 0.102 tyr__L_c + 0.2904 gly_c + 0.1914 thr__L_c + 59.276 atp_c + 0.1647 pro__L_c + 0.0036 damp_c + 0.0007 ergst_c + 0.1339 phe__L_c + 0.2646 val__L_c + 0.2964 leu__L_c + 1.1348 13BDglcn_c + 0.02 so4_c + 0.0447 cmp_c + 0.3018 glu__L_c + 6.0e-6 pa_SC_c + 6.0e-5 pc_SC_c + 0.8079 mannan_c + 0.1854 ser__L_c + 0.0015 zymst_c + 0.1607 arg__L_c + 0.5185 glycogen_c + 1.7e-5 ps_SC_c + 0.1054 gln__L_c + 0.0284 trp__L_c + 0.4588 ala__L_c + 0.0066 cys__L_c + 0.0024 dgmp_c + 0.0036 dtmp_c + 0.1017 asn__L_c + 59.276 h2o_c + 0.2975 asp__L_c + 0.0507 met__L_c = 59.276 adp_c + 59.305 pi_c + 58.7162 h_c
2,1194,THZPSN1_SC,12,1.0 cys__L_c + 1.0 gly_c + 1.0 xu5p__D_c + 1.0 achms_c + 1.0 h_c = 3.0 h2o_c + 1.0 ac_c + 1.0 4mpetz_c + 1.0 co2_c + 1.0 4abut_c + 1.0 nh4_c + 1.0 pyr_c
3,1195,THZPSN2_SC,12,1.0 cys__L_c + 1.0 gly_c + 1.0 r5p_c + 1.0 achms_c + 1.0 h_c = 3.0 h2o_c + 1.0 ac_c + 1.0 4mpetz_c + 1.0 co2_c + 1.0 4abut_c + 1.0 nh4_c + 1.0 pyr_c
4,472,AGAT_SC,11,0.01 1ag3p_SC_c + 0.06 ddcacoa_c + 0.02 dcacoa_c + 0.09 ocdycacoa_c + 0.24 odecoa_c + 0.17 hdcoa_c + 0.05 stcoa_c + 0.27 pmtcoa_c + 0.1 tdcoa_c = 1.0 coa_c + 0.01 pa_SC_c
5,738,FAO141p_even,11,6.0 h2o_x + 6.0 nad_x + 1.0 nadph_x + 6.0 o2_x + 6.0 coa_x + 1.0 tdecoa_x = 6.0 nadh_x + 7.0 accoa_x + 5.0 h_x + 6.0 h2o2_x + 1.0 nadp_x
6,740,FAO161p_even,11,7.0 h2o_x + 7.0 nad_x + 1.0 nadph_x + 1.0 hdcoa_x + 7.0 coa_x + 7.0 o2_x = 7.0 nadh_x + 8.0 accoa_x + 6.0 h_x + 7.0 h2o2_x + 1.0 nadp_x
7,742,FAO181p_even,11,8.0 h2o_x + 8.0 nad_x + 1.0 nadph_x + 8.0 o2_x + 8.0 coa_x + 1.0 odecoa_x = 8.0 nadh_x + 9.0 accoa_x + 7.0 h_x + 8.0 h2o2_x + 1.0 nadp_x
8,744,FAO182p_eveneven,11,8.0 h2o_x + 8.0 nad_x + 2.0 nadph_x + 8.0 o2_x + 8.0 coa_x + 1.0 ocdycacoa_x = 8.0 nadh_x + 9.0 accoa_x + 6.0 h_x + 8.0 h2o2_x + 2.0 nadp_x
9,745,FAO182p_evenodd,11,8.0 h2o_x + 8.0 nad_x + 1.0 nadph_x + 7.0 o2_x + 8.0 coa_x + 1.0 ocdycacoa_x = 8.0 nadh_x + 9.0 accoa_x + 7.0 h_x + 7.0 h2o2_x + 1.0 nadp_x


___

## Summary
In this activity, we accessed a genome-scale metabolic model from the BiGG Models database and analyzed the structure and connectivity of the metabolic network.

> __Key Takeaways:__
>
> 1. **Metabolic models encode biochemical structure**: The stoichiometric matrix provides a mathematical representation of an organism's metabolism, with rows representing metabolites and columns representing reactions. The sign and magnitude of each entry encode whether a metabolite is consumed (negative) or produced (positive) by each reaction.
> 2. **Connectivity reveals metabolic importance**: Highly connected metabolites and reactions are central to an organism's metabolism. Energy metabolites like ATP and cofactors typically show high connectivity, indicating their critical role in linking different metabolic pathways.
> 3. **Network analysis enables biological insight**: By computing connectivity matrices from the binary stoichiometric matrix, we can identify key metabolites and reactions without detailed kinetic information, providing a systems-level view of metabolism useful for metabolic engineering and drug target identification.

___

## Interested in some further reading?
Several publications with a similar theme, i.e., using different matrix factorizations such as [singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition) or different approaches such as looking at the degree distribution, have been published to understand the structural features of biologically derived networks such the stoichiometric arrays:
* [Price ND, Reed JL, Papin JA, Famili I, Palsson BO. Analysis of metabolic capabilities using singular value decomposition of extreme pathway matrices. Biophys J. 2003 Feb;84(2 Pt 1):794-804. doi: 10.1016/S0006-3495(03)74899-1. PMID: 12547764; PMCID: PMC1302660.](https://pubmed.ncbi.nlm.nih.gov/12547764/)
* [Famili I, Palsson BO. Systemic metabolic reactions are obtained by singular value decomposition of genome-scale stoichiometric matrices. J Theor Biol. 2003 Sep 7;224(1):87-96. doi: 10.1016/s0022-5193(03)00146-2. PMID: 12900206.](https://pubmed.ncbi.nlm.nih.gov/12900206/)
* [Barrett CL, Price ND, Palsson BO. Network-level analysis of metabolic regulation in the human red blood cell using random sampling and singular value decomposition. BMC Bioinformatics. 2006 Mar 13;7:132. doi: 10.1186/1471-2105-7-132. PMID: 16533395; PMCID: PMC1421444.](https://pubmed.ncbi.nlm.nih.gov/16533395/)
* [Broido, A.D., Clauset, A. Scale-free networks are rare. Nat Commun 10, 1017 (2019). https://doi.org/10.1038/s41467-019-08746-5](https://rdcu.be/d9Q02)

___